In [1]:
import pandas as pd

df_train = pd.read_parquet("./Data/green_tripdata_2021-01.parquet")

df_test = pd.read_parquet("./Data/green_tripdata_2021-02.parquet")

In [2]:
df_train["duration"] = (
  df_train["lpep_dropoff_datetime"] - df_train["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df_test["duration"] = (
  df_test["lpep_dropoff_datetime"] - df_test["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df_train = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)]
df_test = df_test[(df_test.duration >= 1) & (df_test.duration <= 60)] 

In [3]:
from sklearn.feature_extraction import DictVectorizer

categorical = ["PULocationID", "DOLocationID"]
numerical = ["trip_distance"]

train_dicts = df_train[categorical + numerical].to_dict(orient="records")
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

test_dicts = df_test[categorical + numerical].to_dict(orient="records")
X_test = dv.transform(test_dicts)

y_train = df_train["duration"].values
y_test = df_test["duration"].values

In [4]:
import numpy as np

for X in (X_train, X_test):
  X.indices = X.indices.astype(np.int32)
  X.indptr = X.indptr.astype(np.int32)

In [5]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, root_mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)

print("train", root_mean_squared_error(y_train, model.predict(X_train)))
print("val  ", root_mean_squared_error(y_test, model.predict(X_test)))

train 11.449081717462942
val   12.078673198609463


In [6]:
lasso = Lasso(alpha = 0.1).fit(X_train, y_train)
print("lasso val", root_mean_squared_error(y_test, lasso.predict(X_test)))

lasso val 12.07870203052328
